# Évaluation finale sur le test (notebook 14)

**À exécuter une seule fois.** Ce notebook évalue les 18 modèles sur les 618 radios du test (231 NORMAL, 387 PNEUMONIA, 426 patients présumés) et tranche les quatre hypothèses fixées dans le notebook 12 le 28/09/2026, avant l'entraînement des modèles v3. Les calculs et les règles de lecture ci-dessous sont fixés avant de charger le test. **Ne modifier aucun réglage après avoir vu les résultats** : si le protocole change, le notebook refuse de continuer ; s'il est relancé à l'identique, il réutilise les résultats déjà calculés.

## Les quatre hypothèses (notebook 12)

1. **Question du mémoire :** pour chaque version, écart de F1 macro CBAM − classique, apparié par graine. Aucun sens n'est prédit.
2. **Raccourci :** la spécificité NORMAL des modèles v3 est supérieure à celle des modèles v2 sans augmentation, à architecture et graine égales.
3. **Augmentation :** les modèles v2 avec augmentation font moins de faux positifs que les modèles v2 sans augmentation, à architecture et graine égales.
4. **Mécanisme :** pour les modèles v2, la probabilité de classer une radio NORMAL en PNEUMONIA augmente avec le rapport largeur/hauteur de l'image d'origine ; cette association est plus faible pour v3.

Mesure principale : F1 macro. Mesures secondaires : AUC, rappel PNEUMONIA, spécificité NORMAL. Seuil de décision : classe du plus grand score, sans ajustement.

## Comment chaque hypothèse est calculée (fixé le 28/09/2026, avant de charger le test)

| | Mesure | Cas comparés | Sens prédit |
|---|---|---|---|
| H1 (une par version) | F1 macro CBAM − F1 macro classique | 3 paires (une par graine) | aucun |
| H2 | spécificité NORMAL v3 − v2 sans augmentation | 6 paires (2 architectures × 3 graines) | positif |
| H3 | taux de faux positifs v2 avec − v2 sans augmentation (points de %) | 6 paires | négatif |
| H4, partie 1 | corrélation de Spearman ρ entre le rapport largeur/hauteur d'origine et le score PNEUMONIA (logit PNEUMONIA − logit NORMAL), parmi les radios NORMAL | 12 modèles v2 | positif |
| H4, partie 2 | force de l'association, quel que soit son signe : \|ρ\| v3 − \|ρ\| v2 sans augmentation | 6 paires | négatif |

- **Estimation** : moyenne des paires (ou des modèles), calculée sur tout le test.
- **Intervalle de confiance à 95 %** : rééchantillonnage par patient (bootstrap par groupe), 2 000 tirages. Les patients sont tirés avec remise, séparément dans chaque classe, et toutes les images d'un patient tiré sont gardées. Les mêmes tirages servent pour les 18 modèles : les comparaisons restent appariées. Intervalle = percentiles 2,5 et 97,5.
- **Deux sources d'incertitude, deux conditions** : l'intervalle mesure le hasard des radios du test ; l'accord entre les paires mesure le hasard de l'entraînement (graines).

| Verdict | Règle |
|---|---|
| confirmée | l'intervalle à 95 % est entièrement du côté prédit **et** toutes les paires vont dans le sens prédit |
| partielle | l'estimation va dans le sens prédit, mais une seule de ces deux conditions est remplie |
| non démontrée | l'estimation va dans le sens prédit, mais aucune des deux conditions n'est remplie |
| non confirmée | l'estimation est nulle ou du côté opposé, et l'intervalle contient 0 |
| contredite | l'intervalle à 95 % est entièrement du côté opposé |

Les règles s'appliquent dans cet ordre : un intervalle entièrement du côté opposé donne « contredite », même dans le cas rare où l'estimation serait du côté prédit. Une estimation exactement nulle compte comme « du côté opposé ».

Pour H1, sans sens prédit : **écart démontré** si l'intervalle exclut 0 du côté de l'estimation et que les 3 graines vont dans ce sens ; **écart partiel** si une seule de ces conditions est remplie ; sinon **pas d'écart démontré**. H4 est confirmée dans son ensemble seulement si ses deux parties le sont.

## Limites connues avant les résultats

- Le test a déjà été consulté une fois, au notebook 03, avec les premiers modèles (v1) : leur spécificité s'y était effondrée (0,48 et 0,42). Cette observation a motivé l'analyse des raccourcis. Les 18 modèles évalués ici n'ont jamais vu le test, mais le test n'est plus totalement neuf pour le projet.
- Sept verdicts sont rendus sans correction pour comparaisons multiples : ils se lisent ensemble.
- Trois graines seulement par configuration.
- Les modèles v3 gardent des indices résiduels d'acquisition (netteté, intensité, déformation liée à l'étirement ; notebook 12).

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre, environ 5 à 10 minutes. Avant le test, chaque modèle doit retrouver ses scores déjà enregistrés sur 32 radios de validation. À la fin, un seul fichier est à transmettre : `evaluations/test_final_v2_v3/a_envoyer_notebook_14.zip`. **Garder OneDrive en pause et fermer les CSV du projet dans Excel.**

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time
import zipfile
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from scipy.stats import rankdata, spearmanr
from sklearn import __version__ as version_sklearn
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
DOSSIER_11 = PROJET / "optimisation" / "cartes_attention_v1"
DOSSIER_12 = PROJET / "optimisation" / "v3_sans_raccourci_format"
SORTIE = PROJET / "evaluations" / "test_final_v2_v3"
APPAREIL = torch.device("cpu")
LOT, TAILLE = 16, 224
BANDE_BORDS = 0.15      # régions « centre » et « bords » des caractéristiques d'intensité, comme au notebook 11
N_BOOTSTRAP = 2000
GRAINE_BOOTSTRAP = 20260928
N_CONTROLE = 32         # radios de validation utilisées pour vérifier chaque modèle avant le test
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
NOMS_CLASSES = {0: "NORMAL", 1: "PNEUMONIA"}
GRAINES = [42, 123, 2026]
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
ARCHITECTURES_NOMS = ["CNN_classique", "CNN_CBAM"]
LIBELLES_MODELES = {"CNN_classique": "CNN classique", "CNN_CBAM": "CNN + CBAM"}
V2_SANS, V2_AVEC, V3 = "v2 sans augmentation", "v2 avec augmentation", "v3 sans raccourci"
VERSIONS_MODELES = [V2_SANS, V2_AVEC, V3]
MODELES_ORDRE = [(v, m, g) for v in VERSIONS_MODELES for m in ARCHITECTURES_NOMS for g in GRAINES]
PRETRAITEMENT_V2 = "L; ImageOps.pad 224 bilineaire noir centre; /255"
PRETRAITEMENTS = {
    V2_SANS: PRETRAITEMENT_V2 + "; aucune augmentation",
    V2_AVEC: PRETRAITEMENT_V2,
    V3: "L; resize 224x224 bilineaire sans padding (etirement); ImageOps.equalize; /255; aucune augmentation",
}
# Palette catégorielle validée (daltonisme), identique aux notebooks 12 et 13 : une couleur par version.
COULEURS = {V2_SANS: "#2a78d6", V2_AVEC: "#eb6834", V3: "#1baf7a"}
# Comparaison développement / test : le développement en gris (contexte), le test en violet (série mise en avant).
COULEUR_DEVELOPPEMENT, COULEUR_TEST = "#52514e", "#4a3aa7"
ENCRE, ENCRE_SECONDAIRE, ENCRE_DISCRETE, TRAIT_ZERO = "#0b0b0b", "#52514e", "#898781", "#c3c2b7"
POSITIONS_MODELES = {"CNN_classique": 0, "CNN_CBAM": 1}
DECALAGES = {V2_SANS: -0.22, V2_AVEC: 0.0, V3: 0.22}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def ecrire_npz(chemin, **tableaux):
    temporaire = chemin.with_suffix(".tmp")
    with open(temporaire, "wb") as fichier:
        np.savez_compressed(fichier, **tableaux)
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

def nom_fichier(version, nom, graine):
    prefixe = {V2_SANS: "v2_sans", V2_AVEC: "v2_avec", V3: "v3"}[version]
    return f"{prefixe}__{nom}_graine_{graine}"

def resumer_modeles(tableau, groupes, mesures):
    """Entrée : une ligne par graine. Sortie : moyenne, minimum et maximum sur les graines."""
    resume = tableau.groupby(groupes, sort=False)[mesures].agg(["mean", "min", "max"])
    resume.columns = [f"{mesure}_{statistique}" for mesure, statistique in resume.columns]
    return resume.reset_index()

def styliser(axe):
    axe.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)

for dossier in [SORTIE, SORTIE / "predictions", SORTIE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__)
print("Résultats dans :", SORTIE)

## 1. Préparer le test et les contrôles

Les 618 radios du test sont préparées de deux façons : **v2** (proportions conservées, bandes noires) pour les 12 modèles v2 et **v3** (étirement puis égalisation) pour les 6 modèles v3. Les caractéristiques simples de chaque radio (format d'origine, intensité, netteté) sont calculées avec les mêmes fonctions qu'aux notebooks 11 et 12, pour comparer le test aux données de développement (apprentissage + validation).

Contrôles : effectifs du test, patients et images absents de l'apprentissage et de la validation, et 32 radios de validation préparées pour vérifier chaque modèle avant de l'évaluer sur le test.

In [ ]:
def preparer_v2(image):
    """Préparation v2 (notebooks 03 à 13) : proportions conservées, bandes noires, masque du rectangle de l'image."""
    gris = image.convert("L")
    preparee = ImageOps.pad(gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST, color=0, centering=(0.5, 0.5))
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def etirer(image):
    """Première étape v3 (notebook 12) : niveaux de gris, étirement en 224 × 224 sans bandes noires."""
    return np.array(image.convert("L").resize((TAILLE, TAILLE), Image.Resampling.BILINEAR), dtype=np.uint8)

def egaliser(tableau_u8):
    """Seconde étape v3 (notebook 12) : égalisation d'histogramme."""
    return np.array(ImageOps.equalize(Image.fromarray(np.ascontiguousarray(tableau_u8, dtype=np.uint8))), dtype=np.uint8)

def bornes_regions(masque):
    """Rectangle de l'image et zone centrale, en indices inclus (identique au notebook 11)."""
    lignes = np.flatnonzero(masque.any(axis=1))
    colonnes = np.flatnonzero(masque.any(axis=0))
    exiger(len(lignes) > 0 and len(colonnes) > 0, "Rectangle d'image vide.")
    r0, r1, c0, c1 = int(lignes[0]), int(lignes[-1]), int(colonnes[0]), int(colonnes[-1])
    exiger(bool(masque[r0:r1 + 1, c0:c1 + 1].all()) and int(masque.sum()) == (r1 - r0 + 1) * (c1 - c0 + 1),
           "La zone de l'image n'est pas un rectangle plein.")
    bande_h = int(round(BANDE_BORDS * (r1 - r0 + 1)))
    bande_l = int(round(BANDE_BORDS * (c1 - c0 + 1)))
    cr0, cr1, cc0, cc1 = r0 + bande_h, r1 - bande_h, c0 + bande_l, c1 - bande_l
    exiger(cr0 <= cr1 and cc0 <= cc1, "Zone centrale vide.")
    return {"r0": r0, "r1": r1, "c0": c0, "c1": c1, "cr0": cr0, "cr1": cr1, "cc0": cc0, "cc1": cc1}

def aires(b):
    rectangle = (b["r1"] - b["r0"] + 1) * (b["c1"] - b["c0"] + 1)
    centre = (b["cr1"] - b["cr0"] + 1) * (b["cc1"] - b["cc0"] + 1)
    total = TAILLE * TAILLE
    return {"aire_centre": centre / total, "aire_bords": (rectangle - centre) / total, "aire_padding": (total - rectangle) / total}

def sommes_regions(carte, b):
    total = float(carte.sum())
    rectangle = float(carte[b["r0"]:b["r1"] + 1, b["c0"]:b["c1"] + 1].sum())
    centre = float(carte[b["cr0"]:b["cr1"] + 1, b["cc0"]:b["cc1"] + 1].sum())
    return centre, rectangle - centre, total - rectangle

def caracteristiques_v2(largeur, hauteur, mode, preparee, masque):
    """Mêmes propriétés qu'au notebook 11 (format d'origine, padding, intensité de l'entrée v2)."""
    b = bornes_regions(masque)
    a = aires(b)
    x = preparee.astype(np.float64) / 255.0
    pixels = x[masque]
    p05, p50, p95 = np.percentile(pixels, [5, 50, 95])
    somme_centre, somme_bords, _ = sommes_regions(x, b)
    return {"largeur": largeur, "hauteur": hauteur, "mode_original": mode,
            "rapport_largeur_hauteur": largeur / hauteur, "fraction_padding": a["aire_padding"],
            "moyenne": float(pixels.mean()), "ecart_type": float(pixels.std()),
            "p05": float(p05), "p50": float(p50), "p95": float(p95),
            "moyenne_centre": somme_centre / (a["aire_centre"] * TAILLE * TAILLE),
            "moyenne_bords": somme_bords / (a["aire_bords"] * TAILLE * TAILLE)}

def statistiques_v3(tableau_u8):
    """Mêmes propriétés qu'au notebook 12, calculées sur l'entrée v3."""
    x = tableau_u8.astype(np.float64) / 255.0
    p05, p50, p95 = np.percentile(x, [5, 50, 95])
    nettete = float(np.abs(np.diff(x, axis=0)).mean() + np.abs(np.diff(x, axis=1)).mean())
    return {"v3_moyenne": float(x.mean()), "v3_ecart_type": float(x.std()), "v3_p05": float(p05), "v3_p50": float(p50),
            "v3_p95": float(p95), "v3_nettete": nettete}

# Résultats nécessaires des notebooks précédents.
exiger((DOSSIER_10 / "termine.json").is_file(), "Le notebook 10 v2 n'est pas terminé.")
exiger((DOSSIER_12 / "resultats_v3.csv").is_file(), "Le notebook 12 n'est pas terminé : resultats_v3.csv est absent.")
for fichier in [DOSSIER_11 / "caracteristiques_images.csv", DOSSIER_12 / "caracteristiques_entrees_v3.csv",
                DOSSIER_12 / "mesures_validation_v2_v3.csv"]:
    exiger(fichier.is_file(), f"Fichier introuvable : {fichier}")
PROTOCOLE_10 = json.loads((DOSSIER_10 / "protocole.json").read_text(encoding="utf-8"))
exiger(PROTOCOLE_10.get("version_analyse") == "comparaison_robustesse_augmentation_v2", "Protocole du notebook 10 v2 inattendu.")
exiger(PROTOCOLE_10.get("configurations") == CONFIGURATIONS, "Configurations différentes de celles du notebook 10 v2.")

# Répartition : le test est l'ensemble de test d'origine, dédoublonné (notebook 02).
lecture = {"dtype": {"groupe": str, "empreinte": str}}
train_df = pd.read_csv(REPARTITION / "train.csv", **lecture)
val_df = pd.read_csv(REPARTITION / "val.csv", **lecture)
test_df = pd.read_csv(REPARTITION / "test.csv", **lecture)
exiger(test_df["classe"].value_counts().to_dict() == {"NORMAL": 231, "PNEUMONIA": 387} and len(test_df) == 618,
       "Effectifs du test inattendus.")
exiger(test_df["chemin_relatif"].is_unique and test_df["groupe"].notna().all(), "Chemins dupliqués ou patients inconnus dans le test.")
for colonne in ["groupe", "empreinte", "chemin_relatif"]:
    exiger(set(test_df[colonne]).isdisjoint(train_df[colonne]) and set(test_df[colonne]).isdisjoint(val_df[colonne]),
           f"Le test partage des valeurs de « {colonne} » avec l'apprentissage ou la validation.")
exiger((test_df.groupby("groupe")["classe"].nunique() == 1).all(), "Un patient du test a des radios des deux classes.")
N_TEST = len(test_df)
N_PATIENTS = int(test_df["groupe"].nunique())
Y_TEST = test_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
SHA_TEST_CSV = empreinte_fichier(REPARTITION / "test.csv")

# Modèles v3 du notebook 12.
RESULTATS_V3 = pd.read_csv(DOSSIER_12 / "resultats_v3.csv")
exiger(len(RESULTATS_V3) == 6 and set(zip(RESULTATS_V3["modele"], RESULTATS_V3["graine"].astype(int)))
       == {(m, g) for m in ARCHITECTURES_NOMS for g in GRAINES}, "resultats_v3.csv doit contenir les six modèles v3.")
for ligne in RESULTATS_V3.to_dict("records"):
    reglages = CONFIGURATIONS[ligne["modele"]]
    # Tolérance : la relecture d'un CSV peut modifier le dernier chiffre d'un nombre décimal.
    exiger(int(ligne["indice"]) == reglages["indice"] and abs(float(ligne["lr"]) - reglages["lr"]) < 1e-12
           and abs(float(ligne["dropout"]) - reglages["dropout"]) < 1e-12,
           f"Réglages v3 inattendus : {ligne['modele']}, graine {ligne['graine']}.")

def dossier_v3(nom, graine):
    return DOSSIER_12 / nom / f"graine_{graine}" / f"configuration_{CONFIGURATIONS[nom]['indice']:02d}"

# Préparation du test (v2 et v3) et caractéristiques de chaque radio.
PREPAREES_TEST = np.empty((N_TEST, TAILLE, TAILLE), dtype=np.uint8)
EGALISEES_TEST = np.empty((N_TEST, TAILLE, TAILLE), dtype=np.uint8)
lignes = []
empreinte_v2, empreinte_v3 = hashlib.sha256(), hashlib.sha256()
for i, ligne in enumerate(tqdm(test_df.to_dict("records"), desc="Préparation du test")):
    with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
        largeur, hauteur = image.size
        mode = image.mode
        preparee, masque = preparer_v2(image)
        etiree = etirer(image)
    exiger(masque.any() and (preparee[~masque] == 0).all(), "Erreur de géométrie du padding.")
    if {"largeur", "hauteur"}.issubset(test_df.columns):
        exiger((largeur, hauteur) == (int(ligne["largeur"]), int(ligne["hauteur"])),
               f"Taille différente de l'inventaire du notebook 02 : {ligne['chemin_relatif']}")
    PREPAREES_TEST[i] = preparee
    EGALISEES_TEST[i] = egaliser(etiree)
    empreinte_v2.update(ligne["chemin_relatif"].encode("utf-8") + b"\0")
    empreinte_v2.update(PREPAREES_TEST[i].tobytes())
    empreinte_v3.update(ligne["chemin_relatif"].encode("utf-8") + b"\0")
    empreinte_v3.update(EGALISEES_TEST[i].tobytes())
    lignes.append({"ensemble": "test", "chemin_relatif": ligne["chemin_relatif"], "groupe": ligne["groupe"],
                   "classe": ligne["classe"], **caracteristiques_v2(largeur, hauteur, mode, preparee, masque),
                   **statistiques_v3(EGALISEES_TEST[i])})
CARACTERISTIQUES_TEST = pd.DataFrame(lignes)
RAPPORTS_TEST = CARACTERISTIQUES_TEST["rapport_largeur_hauteur"].to_numpy(dtype=np.float64)
SHA_TEST_V2, SHA_TEST_V3 = empreinte_v2.hexdigest(), empreinte_v3.hexdigest()
ecrire_csv(CARACTERISTIQUES_TEST, SORTIE / "caracteristiques_test.csv")

# 32 radios de validation pour vérifier chaque modèle avant le test.
CONTROLE = val_df.head(N_CONTROLE)
IDENTIFIANTS_CONTROLE = CONTROLE["chemin_relatif"].tolist()
PREPAREES_CONTROLE = np.empty((N_CONTROLE, TAILLE, TAILLE), dtype=np.uint8)
EGALISEES_CONTROLE = np.empty((N_CONTROLE, TAILLE, TAILLE), dtype=np.uint8)
for i, identifiant in enumerate(IDENTIFIANTS_CONTROLE):
    with Image.open(IMAGES / identifiant) as image:
        PREPAREES_CONTROLE[i] = preparer_v2(image)[0]
        EGALISEES_CONTROLE[i] = egaliser(etirer(image))

def fichier_reference(version, nom, graine):
    """Prédictions de validation déjà enregistrées (notebooks 08 v2, 10 v2 et 12)."""
    if version == V2_SANS:
        return DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__originale.csv"
    if version == V2_AVEC:
        return DOSSIER_10 / "predictions" / f"avec__{nom}_graine_{graine}__originale.csv"
    return DOSSIER_12 / "predictions_validation" / f"{nom}_graine_{graine}.csv"

def reference_controle(version, nom, graine):
    chemin = fichier_reference(version, nom, graine)
    exiger(chemin.is_file(), f"Prédictions de référence introuvables : {chemin}")
    tableau = pd.read_csv(chemin, dtype={"groupe": str}, nrows=N_CONTROLE)
    exiger(tableau["chemin_relatif"].tolist() == IDENTIFIANTS_CONTROLE, f"Ordre des images différent : {chemin}")
    return tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)

REFERENCES_CONTROLE = {cle: reference_controle(*cle) for cle in MODELES_ORDRE}

def controler_logits(calcules, attendus, contexte):
    ecart = float(np.abs(np.asarray(calcules, dtype=np.float64) - attendus).max())
    tolerance = 1e-4 + 1e-5 * float(np.abs(attendus).max())
    exiger(ecart <= tolerance, f"{contexte} : scores bruts différents des prédictions de référence (écart maximal {ecart:.2e}). "
                               "Vérifier le noyau, les versions et les sauvegardes avant de continuer.")
    return ecart

def entrees(version, source_v2, source_v3):
    return ((source_v3 if version == V3 else source_v2).astype(np.float32) / 255.0)[:, None]

def controler_modeles():
    """Avant tout calcul sur le test : chaque modèle doit retrouver ses scores de validation déjà enregistrés."""
    ecarts = {}
    for cle in MODELES_ORDRE:
        controle = entrees(cle[0], PREPAREES_CONTROLE, EGALISEES_CONTROLE)
        logits = np.concatenate([logits_lot(MODELES[cle], controle[d:d + LOT]) for d in range(0, N_CONTROLE, LOT)])
        ecarts[cle] = controler_logits(logits, REFERENCES_CONTROLE[cle], f"{nom_fichier(*cle)}, contrôle sur la validation")
    return ecarts
print(f"Test : {N_TEST} radios ({(Y_TEST == 0).sum()} NORMAL, {(Y_TEST == 1).sum()} PNEUMONIA), {N_PATIENTS} patients présumés, "
      "aucun en commun avec l'apprentissage ou la validation.")
print(f"Rapport largeur/hauteur moyen dans le test : NORMAL {RAPPORTS_TEST[Y_TEST == 0].mean():.2f}, "
      f"PNEUMONIA {RAPPORTS_TEST[Y_TEST == 1].mean():.2f}.")

In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # Connexion résiduelle, identique aux notebooks 06 v2 à 13.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
VERSION_DE_VARIANTE = {"sans": V2_SANS, "avec": V2_AVEC}

def charger_modele(version, nom, graine, chemin):
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {version}, {nom}.")
    exiger(protocole["graine"] == graine, f"Graine incompatible : {version}, {nom}, {graine}.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == PRETRAITEMENTS[version],
           f"Classes ou préparation incompatibles : {version}, {nom}, graine {graine}.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], f"Sauvegarde d'une autre configuration : {version}, {nom}, graine {graine}.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    return modele, int(etat["epoque"])

MODELES, INFORMATIONS = {}, []
exiger(len(PROTOCOLE_10["modeles"]) == 12, "Le protocole du notebook 10 v2 doit lister douze modèles.")
for info in PROTOCOLE_10["modeles"]:
    version, nom, graine = VERSION_DE_VARIANTE[info["variante"]], info["modele"], int(info["graine"])
    chemin = PROJET.joinpath(*info["sauvegarde_relative"].split("/"))
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    exiger(empreinte_fichier(chemin) == info["sha256_sauvegarde"], f"La sauvegarde {chemin} a changé depuis le notebook 10 v2.")
    modele, epoque = charger_modele(version, nom, graine, chemin)
    exiger(epoque == int(info["epoque"]), "Époque incompatible avec le notebook 10 v2.")
    MODELES[(version, nom, graine)] = modele
    INFORMATIONS.append({"version": version, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": info["sha256_sauvegarde"], "sauvegarde_relative": info["sauvegarde_relative"]})
for ligne in RESULTATS_V3.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    chemin = dossier_v3(nom, graine) / "meilleur.pt"
    exiger(chemin.is_file(), f"Sauvegarde v3 introuvable : {chemin}")
    modele, epoque = charger_modele(V3, nom, graine, chemin)
    exiger(epoque == int(ligne["epoque_retenue"]), f"Époque v3 incompatible avec resultats_v3.csv : {nom}, graine {graine}.")
    MODELES[(V3, nom, graine)] = modele
    INFORMATIONS.append({"version": V3, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": empreinte_fichier(chemin), "sauvegarde_relative": chemin.relative_to(PROJET).as_posix()})
exiger(set(MODELES) == set(MODELES_ORDRE), "Il manque des modèles.")

def logits_lot(modele, x):
    """Logits (B, 2) pour un lot numpy de forme (B, 1, 224, 224)."""
    with torch.no_grad():
        sortie = modele(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(APPAREIL))
    return sortie.cpu().numpy().astype(np.float64)

# Contrôle des 18 modèles sur 32 radios de validation, AVANT d'enregistrer le protocole et de toucher au test.
ECARTS_CONTROLE = controler_modeles()
print("18 modèles chargés et figés : 12 modèles v2 (notebook 10 v2) et 6 modèles v3 (notebook 12).")
print(f"Contrôle sur la validation réussi pour les 18 modèles (écart maximal aux scores enregistrés : {max(ECARTS_CONTROLE.values()):.1e}).")

In [ ]:
HYPOTHESES = {
    "H1": "Question du mémoire : pour chaque version, écart de F1 macro CBAM − classique, apparié par graine. Aucun sens n'est prédit.",
    "H2": "Raccourci : la spécificité NORMAL des modèles v3 est supérieure à celle des modèles v2 sans augmentation, à architecture et graine égales.",
    "H3": "Augmentation : les modèles v2 avec augmentation font moins de faux positifs que les modèles v2 sans augmentation, à architecture et graine égales.",
    "H4": "Mécanisme : pour les modèles v2, la probabilité de classer une radio NORMAL en PNEUMONIA augmente avec le rapport largeur/hauteur de l'image d'origine ; cette association est plus faible pour v3.",
}
OPERATIONNALISATION = {
    "H1": "par version : moyenne sur 3 graines de F1 macro(CBAM) - F1 macro(classique); aucun sens predit",
    "H2": "moyenne sur 6 paires (architecture, graine) de specificite(v3) - specificite(v2 sans); sens predit positif",
    "H3": "moyenne sur 6 paires de taux de faux positifs en %(v2 avec) - (v2 sans); sens predit negatif",
    "H4 partie 1": "moyenne sur les 12 modeles v2 de rho de Spearman(rapport largeur/hauteur d'origine, logit PNEUMONIA - logit NORMAL) parmi les radios NORMAL; sens predit positif",
    "H4 partie 2": "moyenne sur 6 paires de |rho(v3)| - |rho(v2 sans)| (force de l'association, quel que soit son signe); sens predit negatif",
}
REGLES = {
    "confirmee": "IC95 entierement du cote predit ET toutes les paires dans le sens predit",
    "partielle": "estimation du cote predit, une seule des deux conditions remplie",
    "non demontree": "estimation du cote predit, aucune des deux conditions",
    "non confirmee": "estimation nulle ou du cote oppose, IC95 contenant 0",
    "contredite": "IC95 entierement du cote oppose",
    "H1": "ecart demontre si IC95 exclut 0 et 3 graines de meme signe; ecart partiel si une seule condition; sinon pas d'ecart demontre",
}
PROTOCOLE = {
    "version_analyse": "evaluation_finale_test_v1", "ensemble": "test", "effectif": N_TEST, "patients": N_PATIENTS,
    "sha256_test_csv": SHA_TEST_CSV, "sha256_entrees_test_v2": SHA_TEST_V2, "sha256_entrees_test_v3": SHA_TEST_V3,
    "hypotheses": HYPOTHESES, "hypotheses_fixees": "notebook 12, le 28/09/2026, avant l'entrainement v3",
    "operationnalisation_et_regles_fixees": "notebook 14, le 28/09/2026, avant le chargement du test",
    "operationnalisation": OPERATIONNALISATION, "regles": REGLES,
    "seuil": "classe du plus grand logit, NORMAL en cas d'egalite",
    "bootstrap": {"tirages": N_BOOTSTRAP, "graine": GRAINE_BOOTSTRAP, "generateur": "numpy PCG64",
                  "unite": "groupe (patient presume, notebook 02)", "stratification": "par classe, nombre de patients de chaque classe conserve",
                  "intervalle": "percentiles 2,5 et 97,5"},
    "controle_avant_test": f"logits des {N_CONTROLE} premieres radios de validation compares aux predictions de reference",
    "modeles": INFORMATIONS, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "limite": "Test deja consulte au notebook 03 avec les modeles v1; 7 verdicts sans correction multiple; 3 graines; indices residuels en v3.",
}
ID_PROTOCOLE = hashlib.sha256(json.dumps(PROTOCOLE, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
exiger(set(globals().get("ECARTS_CONTROLE", {})) == set(MODELES_ORDRE),
       "Exécuter d'abord la cellule des modèles : les 18 contrôles sur la validation doivent réussir avant d'enregistrer le protocole.")
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole, les modèles, les images ou les versions des bibliothèques ont changé depuis la première évaluation "
           "du test. Ne rien supprimer : conserver les premiers résultats du dossier test_final_v2_v3.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)

def verdict(valeurs, bas, haut, sens):
    """Verdict selon les règles fixées en tête du notebook. sens : +1, -1 ou 0 (aucun sens prédit)."""
    valeurs = np.asarray(valeurs, dtype=float)
    estimation = float(np.round(valeurs.mean(), 12))
    if sens == 0:
        if estimation == 0:
            return "pas d'écart démontré"
        signe = np.sign(estimation)
        intervalle = (bas > 0) if estimation > 0 else (haut < 0)
        unanime = bool((np.sign(valeurs) == signe).all())
        cote = "CBAM" if estimation > 0 else "classique"
        if intervalle and unanime:
            return f"écart démontré, en faveur du {cote}"
        if intervalle or unanime:
            return f"écart partiel, en faveur du {cote}"
        return "pas d'écart démontré"
    intervalle_predit = (bas > 0) if sens > 0 else (haut < 0)
    intervalle_oppose = (haut < 0) if sens > 0 else (bas > 0)
    unanime = bool((sens * valeurs > 0).all())
    if intervalle_oppose:
        return "contredite"
    if sens * estimation <= 0:
        return "non confirmée"
    if intervalle_predit and unanime:
        return "confirmée"
    if intervalle_predit or unanime:
        return "partielle"
    return "non démontrée"

print("Protocole enregistré :", fichier_protocole)

## 2. Évaluer les 18 modèles sur le test

Les 18 modèles ont déjà réussi le contrôle sur 32 radios de validation (cellule des modèles). Chacun calcule maintenant ses scores sur les 618 radios du test. Les prédictions de chaque modèle sont enregistrées dans `predictions/` ; si le notebook est relancé, elles sont relues et non recalculées.

Les performances de validation (notebook 12) sont rappelées à côté : un grand écart entre validation et test signale une mauvaise généralisation.

In [ ]:
def evaluer(version, nom, graine):
    cle = nom_fichier(version, nom, graine)
    fichier = SORTIE / "predictions" / f"{cle}.csv"
    if fichier.exists():
        tableau = pd.read_csv(fichier, dtype={"groupe": str})
        exiger(len(tableau) == N_TEST and (tableau["id_protocole"] == ID_PROTOCOLE).all()
               and tableau["chemin_relatif"].tolist() == test_df["chemin_relatif"].tolist(),
               f"{fichier.name} : résultats existants incompatibles avec ce protocole.")
        return tableau, True
    modele = MODELES[(version, nom, graine)]
    ecart = ECARTS_CONTROLE[(version, nom, graine)]
    x = entrees(version, PREPAREES_TEST, EGALISEES_TEST)
    logits = np.concatenate([logits_lot(modele, x[d:d + LOT]) for d in range(0, N_TEST, LOT)])
    exiger(logits.shape == (N_TEST, 2) and np.isfinite(logits).all(), f"{cle} : scores du test invalides.")
    tableau = test_df[["chemin_relatif", "groupe"]].copy()
    tableau.insert(0, "graine", graine)
    tableau.insert(0, "modele", nom)
    tableau.insert(0, "version", version)
    tableau["classe_vraie"] = Y_TEST
    tableau["rapport_largeur_hauteur"] = RAPPORTS_TEST
    tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
    tableau["score_pneumonie"] = logits[:, 1] - logits[:, 0]
    tableau["prediction"] = (logits[:, 1] > logits[:, 0]).astype(int)
    tableau["ecart_controle_validation"] = ecart
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_csv(tableau, fichier)
    return tableau, False

def mesures_test(y, score, rapports):
    prediction = (score > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"f1_macro": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
            "auc_roc": roc_auc_score(y, score), "rappel_pneumonie": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
            "precision_pneumonie": tp / (tp + fp) if tp + fp > 0 else 0.0, "exactitude": (tn + tp) / len(y),
            "taux_faux_positifs_pct": 100 * fp / (tn + fp), "faux_positifs": int(fp), "faux_negatifs": int(fn),
            "rho_format_score_NORMAL": float(spearmanr(rapports[y == 0], score[y == 0])[0])}

PREDICTIONS, lignes = {}, []
for cle in MODELES_ORDRE:
    debut = time.perf_counter()
    tableau, reutilise = evaluer(*cle)
    PREDICTIONS[cle] = tableau
    etat = "réutilisé" if reutilise else f"calculé en {time.perf_counter() - debut:.0f} s"
    print(f"{cle[0]:<21} | {cle[1]:<13} | graine {cle[2]:>4} | contrôle validation : écart {tableau['ecart_controle_validation'].iat[0]:.1e} | {etat}")
    lignes.append({"version": cle[0], "modele": cle[1], "graine": cle[2],
                   **mesures_test(Y_TEST, tableau["score_pneumonie"].to_numpy(dtype=np.float64), RAPPORTS_TEST)})
MESURES_TEST = pd.DataFrame(lignes)
ecrire_csv(MESURES_TEST, SORTIE / "mesures_test_par_modele.csv")

METRIQUES = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "precision_pneumonie", "exactitude",
             "faux_positifs", "faux_negatifs", "rho_format_score_NORMAL"]
SYNTHESE_TEST = resumer_modeles(MESURES_TEST, ["version", "modele"], METRIQUES)

# Rappel de la validation (notebook 12) et écart test − validation, modèle par modèle.
validation = pd.read_csv(DOSSIER_12 / "mesures_validation_v2_v3.csv")
comparees = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL"]
VALIDATION_TEST = MESURES_TEST[["version", "modele", "graine"] + comparees].merge(
    validation[["version", "modele", "graine"] + comparees], on=["version", "modele", "graine"],
    suffixes=("_test", "_validation"), validate="one_to_one")
exiger(len(VALIDATION_TEST) == 18, "Mesures de validation du notebook 12 incomplètes.")
for m in comparees:
    VALIDATION_TEST[f"ecart_{m}"] = VALIDATION_TEST[f"{m}_test"] - VALIDATION_TEST[f"{m}_validation"]
RESUME_VALIDATION_TEST = resumer_modeles(VALIDATION_TEST, ["version", "modele"],
                                         [f"{m}_validation" for m in comparees] + [f"{m}_test" for m in comparees]
                                         + [f"ecart_{m}" for m in comparees])
ecrire_csv(SYNTHESE_TEST, SORTIE / "synthese_test.csv")
ecrire_csv(VALIDATION_TEST, SORTIE / "validation_et_test_par_modele.csv")
ecrire_csv(RESUME_VALIDATION_TEST, SORTIE / "resume_validation_et_test.csv")

print("\nPerformances sur le test (moyenne, minimum et maximum sur 3 graines) :")
display(SYNTHESE_TEST[["version", "modele", "f1_macro_mean", "f1_macro_min", "f1_macro_max", "auc_roc_mean",
                       "rappel_pneumonie_mean", "specificite_NORMAL_mean", "faux_positifs_mean", "faux_negatifs_mean"]].round(4))
print("Validation (notebook 12) et test, moyenne sur 3 graines :")
display(RESUME_VALIDATION_TEST[["version", "modele", "f1_macro_validation_mean", "f1_macro_test_mean", "ecart_f1_macro_mean",
                                "specificite_NORMAL_validation_mean", "specificite_NORMAL_test_mean"]].round(4))

## 3. Incertitude et verdicts des hypothèses

Le rééchantillonnage par patient est fait une fois pour toutes (2 000 tirages, graine fixée) et enregistré dans `bootstrap.npz`. Pour aller vite, les mesures de chaque tirage sont calculées par une fonction simple ; elle doit d'abord retrouver exactement, sur le test complet, les valeurs calculées avec scikit-learn et SciPy.

Les tableaux donnent, pour chaque hypothèse : l'estimation, l'intervalle de confiance à 95 %, le nombre de paires allant dans le sens prédit et le verdict selon les règles fixées en tête du notebook. Des tableaux de détail (mesures secondaires de H1, résultats par architecture) servent à décrire les résultats ; ils ne donnent pas de verdict.

In [ ]:
METRIQUES_BOOT = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "taux_faux_positifs_pct", "rho_format_score_NORMAL"]
INDEX = {cle: k for k, cle in enumerate(MODELES_ORDRE)}
SCORES = np.stack([PREDICTIONS[cle]["score_pneumonie"].to_numpy(dtype=np.float64) for cle in MODELES_ORDRE])

def mesures_rapides(y, score, rangs_rapport_normal):
    """Mêmes mesures que mesures_test, en version rapide pour le rééchantillonnage."""
    positif, predit = y == 1, score > 0
    tp, fn = int(np.sum(predit & positif)), int(np.sum(~predit & positif))
    fp, tn = int(np.sum(predit & ~positif)), int(np.sum(~predit & ~positif))
    f1_pneumonie = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0.0
    f1_normal = 2 * tn / (2 * tn + fn + fp) if 2 * tn + fn + fp > 0 else 0.0
    n1, n0 = tp + fn, tn + fp
    rangs = rankdata(score)
    auc = (rangs[positif].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    rho = np.corrcoef(rangs_rapport_normal, rankdata(score[~positif]))[0, 1]
    return (0.5 * (f1_pneumonie + f1_normal), auc, tp / n1, tn / n0, 100 * fp / n0, rho)

# Contrôle : la fonction rapide retrouve les valeurs de scikit-learn et SciPy sur le test complet.
rangs_complets = rankdata(RAPPORTS_TEST[Y_TEST == 0])
for cle in MODELES_ORDRE:
    rapides = mesures_rapides(Y_TEST, SCORES[INDEX[cle]], rangs_complets)
    reference = MESURES_TEST.iloc[INDEX[cle]]
    for nom_mesure, valeur in zip(METRIQUES_BOOT, rapides):
        exiger(abs(valeur - float(reference[nom_mesure])) < 1e-9, f"Mesure rapide incorrecte : {nom_mesure}, {cle}.")

fichier_bootstrap = SORTIE / "bootstrap.npz"
if fichier_bootstrap.exists():
    with np.load(fichier_bootstrap) as donnees:
        exiger(str(donnees["id_protocole"]) == ID_PROTOCOLE and donnees["distribution"].shape == (N_BOOTSTRAP, 18, len(METRIQUES_BOOT))
               and list(donnees["metriques"]) == METRIQUES_BOOT
               and list(donnees["modeles"]) == ["|".join(map(str, cle)) for cle in MODELES_ORDRE],
               "bootstrap.npz ne correspond pas à ce protocole.")
        BOOT = donnees["distribution"]
    print("Rééchantillonnage réutilisé.")
else:
    groupes = test_df["groupe"].to_numpy()
    uniques = pd.unique(groupes)
    indices_groupe = {g: np.flatnonzero(groupes == g) for g in uniques}
    par_classe = {c: [indices_groupe[g] for g in uniques if Y_TEST[indices_groupe[g][0]] == c] for c in [0, 1]}
    generateur = np.random.Generator(np.random.PCG64(GRAINE_BOOTSTRAP))
    BOOT = np.empty((N_BOOTSTRAP, 18, len(METRIQUES_BOOT)), dtype=np.float64)
    for b in tqdm(range(N_BOOTSTRAP), desc="Rééchantillonnage par patient"):
        morceaux = []
        for c in [0, 1]:
            tires = generateur.integers(0, len(par_classe[c]), size=len(par_classe[c]))
            morceaux.extend(par_classe[c][k] for k in tires)
        idx = np.concatenate(morceaux)
        y = Y_TEST[idx]
        rangs_rapport = rankdata(RAPPORTS_TEST[idx][y == 0])
        for k in range(18):
            BOOT[b, k] = mesures_rapides(y, SCORES[k, idx], rangs_rapport)
    exiger(np.isfinite(BOOT).all(), "Valeurs non finies dans le rééchantillonnage.")
    ecrire_npz(fichier_bootstrap, distribution=BOOT, id_protocole=np.array(ID_PROTOCOLE),
               modeles=np.array(["|".join(map(str, cle)) for cle in MODELES_ORDRE]), metriques=np.array(METRIQUES_BOOT))
    print(f"Rééchantillonnage terminé : {N_BOOTSTRAP} tirages de {N_PATIENTS} patients.")

def estimer(mesure, cas, absolu=False):
    """cas : liste de paires (a, b) pour une différence a − b, ou de modèles seuls. absolu : comparer les valeurs absolues.
    Les résultats sont arrondis à 12 décimales : une égalité exacte ne doit pas devenir ±1e-16 et changer un verdict."""
    k = METRIQUES_BOOT.index(mesure)
    transformer = np.abs if absolu else (lambda z: z)
    def valeur(cle):
        return transformer(float(MESURES_TEST.iloc[INDEX[cle]][mesure]))
    if isinstance(cas[0][0], tuple):
        valeurs = np.array([valeur(a) - valeur(b) for a, b in cas])
        distribution = np.mean([transformer(BOOT[:, INDEX[a], k]) - transformer(BOOT[:, INDEX[b], k]) for a, b in cas], axis=0)
    else:
        valeurs = np.array([valeur(a) for a in cas])
        distribution = np.mean([transformer(BOOT[:, INDEX[a], k]) for a in cas], axis=0)
    valeurs, distribution = np.round(valeurs, 12), np.round(distribution, 12)
    bas, haut = np.round(np.percentile(distribution, [2.5, 97.5]), 12)
    return valeurs, float(np.round(valeurs.mean(), 12)), float(bas), float(haut)

def ligne_resultat(code_h, comparaison, mesure, unite, cas, sens, echelle=1.0, avec_verdict=True, absolu=False):
    valeurs, estimation, bas, haut = estimer(mesure, cas, absolu)
    # Sens de référence pour compter les cas : le sens prédit, ou celui de l'estimation pour H1.
    reference = sens if sens != 0 else int(np.sign(estimation))
    ligne = {"hypothese": code_h, "comparaison": comparaison, "mesure": mesure, "unite": unite,
             "sens_predit": {1: "positif", -1: "négatif", 0: "aucun"}[sens],
             "estimation": estimation * echelle, "ic95_bas": bas * echelle, "ic95_haut": haut * echelle,
             "cas_dans_le_sens": int(np.sum(np.sign(valeurs) == reference)) if reference != 0 else 0,
             "cas": len(valeurs), "valeurs_par_cas": " ; ".join(f"{v * echelle:+.4f}" for v in valeurs)}
    if avec_verdict:
        ligne["verdict"] = verdict(valeurs, bas, haut, sens)
    return ligne

def paires(version_a, modele_a, version_b, modele_b, architectures=ARCHITECTURES_NOMS):
    if modele_a is None:
        return [((version_a, m, g), (version_b, m, g)) for m in architectures for g in GRAINES]
    return [((version_a, modele_a, g), (version_b, modele_b, g)) for g in GRAINES]

HYPOTHESES_RESULTATS = pd.DataFrame(
    [ligne_resultat("H1", f"{v} : CBAM − classique", "f1_macro", "F1 macro", paires(v, "CNN_CBAM", v, "CNN_classique"), 0)
     for v in VERSIONS_MODELES]
    + [ligne_resultat("H2", "spécificité : v3 − v2 sans augmentation", "specificite_NORMAL", "points de %",
                      paires(V3, None, V2_SANS, None), +1, echelle=100),
       ligne_resultat("H3", "faux positifs : v2 avec − v2 sans augmentation", "taux_faux_positifs_pct", "points de %",
                      paires(V2_AVEC, None, V2_SANS, None), -1),
       ligne_resultat("H4 partie 1", "ρ format → score parmi les NORMAL, 12 modèles v2", "rho_format_score_NORMAL", "ρ de Spearman",
                      [(v, m, g) for v in [V2_SANS, V2_AVEC] for m in ARCHITECTURES_NOMS for g in GRAINES], +1),
       ligne_resultat("H4 partie 2", "|ρ| : v3 − v2 sans augmentation", "rho_format_score_NORMAL", "|ρ| de Spearman",
                      paires(V3, None, V2_SANS, None), -1, absolu=True)])
h4 = HYPOTHESES_RESULTATS[HYPOTHESES_RESULTATS["hypothese"].str.startswith("H4")]["verdict"]
VERDICT_H4 = "confirmée" if (h4 == "confirmée").all() else "non confirmée dans son ensemble (voir les deux parties)"

# Détails descriptifs, sans verdict : mesures secondaires de H1 et résultats par architecture pour H2 à H4.
details = []
for v in VERSIONS_MODELES:
    for mesure, echelle, unite in [("auc_roc", 1.0, "AUC"), ("rappel_pneumonie", 100, "points de %"), ("specificite_NORMAL", 100, "points de %")]:
        details.append(ligne_resultat("H1 (secondaire)", f"{v} : CBAM − classique", mesure, unite,
                                      paires(v, "CNN_CBAM", v, "CNN_classique"), 0, echelle=echelle, avec_verdict=False))
for nom in ARCHITECTURES_NOMS:
    libelle = LIBELLES_MODELES[nom]
    details.append(ligne_resultat("H2 (détail)", f"{libelle} : spécificité v3 − v2 sans", "specificite_NORMAL", "points de %",
                                  paires(V3, None, V2_SANS, None, [nom]), +1, echelle=100, avec_verdict=False))
    details.append(ligne_resultat("H3 (détail)", f"{libelle} : faux positifs v2 avec − v2 sans", "taux_faux_positifs_pct", "points de %",
                                  paires(V2_AVEC, None, V2_SANS, None, [nom]), -1, avec_verdict=False))
    for v in [V2_SANS, V2_AVEC, V3]:
        # Un sens n'est prédit que pour les modèles v2 ; pour v3, H4 porte sur la force de l'association.
        details.append(ligne_resultat("H4 (détail)", f"{libelle} : ρ format → score, {v}", "rho_format_score_NORMAL", "ρ de Spearman",
                                      [(v, nom, g) for g in GRAINES], +1 if v != V3 else 0, avec_verdict=False))
DETAILS_HYPOTHESES = pd.DataFrame(details)

# Intervalle de confiance de chaque mesure, moyenne sur les 3 graines, pour chaque version et architecture.
lignes = []
for v in VERSIONS_MODELES:
    for nom in ARCHITECTURES_NOMS:
        for mesure in METRIQUES_BOOT:
            valeurs, estimation, bas, haut = estimer(mesure, [(v, nom, g) for g in GRAINES])
            lignes.append({"version": v, "modele": nom, "mesure": mesure, "moyenne_3_graines": estimation,
                           "ic95_bas": bas, "ic95_haut": haut, "minimum_graines": valeurs.min(), "maximum_graines": valeurs.max()})
INTERVALLES_TEST = pd.DataFrame(lignes)

BILAN_HYPOTHESES = HYPOTHESES_RESULTATS[["hypothese", "comparaison", "unite", "sens_predit", "estimation", "ic95_bas", "ic95_haut",
                                         "cas_dans_le_sens", "cas", "verdict"]].copy()
ecrire_csv(HYPOTHESES_RESULTATS, SORTIE / "hypotheses_resultats.csv")
ecrire_csv(BILAN_HYPOTHESES, SORTIE / "bilan_hypotheses.csv")
ecrire_csv(DETAILS_HYPOTHESES, SORTIE / "details_hypotheses.csv")
ecrire_csv(INTERVALLES_TEST, SORTIE / "intervalles_test.csv")

pd.set_option("display.max_colwidth", 80)
print("Verdicts des hypothèses (règles fixées avant le test) :")
display(BILAN_HYPOTHESES.round(4))
print("H4 dans son ensemble :", VERDICT_H4)
print("Mesures secondaires de H1 et détails par architecture (descriptifs) :")
display(DETAILS_HYPOTHESES[["hypothese", "comparaison", "unite", "estimation", "ic95_bas", "ic95_haut", "valeurs_par_cas"]].round(4))

In [ ]:
# Figure 1 : performances sur le test (points : graines ; trait : moyenne ; barre verticale : IC 95 % de la moyenne).
panneaux = [("f1_macro", "F1 macro"), ("auc_roc", "AUC"), ("rappel_pneumonie", "Rappel PNEUMONIA"), ("specificite_NORMAL", "Spécificité NORMAL")]
fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
for axe, (mesure, titre) in zip(axes.ravel(), panneaux):
    for version in VERSIONS_MODELES:
        for nom, x in POSITIONS_MODELES.items():
            position = x + DECALAGES[version]
            y = MESURES_TEST[(MESURES_TEST["version"] == version) & (MESURES_TEST["modele"] == nom)][mesure].to_numpy(dtype=float)
            ic = INTERVALLES_TEST[(INTERVALLES_TEST["version"] == version) & (INTERVALLES_TEST["modele"] == nom)
                                  & (INTERVALLES_TEST["mesure"] == mesure)].iloc[0]
            axe.vlines(position, ic["ic95_bas"], ic["ic95_haut"], color=COULEURS[version], linewidth=1.2, alpha=0.7, zorder=1)
            axe.scatter(np.full(len(y), position), y, s=64, color=COULEURS[version], edgecolors="white", linewidths=1.5,
                        zorder=3, label=version if nom == "CNN_classique" else None)
            axe.hlines(y.mean(), position - 0.08, position + 0.08, color=COULEURS[version], linewidth=2, zorder=2)
    axe.set_xticks(list(POSITIONS_MODELES.values()), [LIBELLES_MODELES[m] for m in POSITIONS_MODELES])
    axe.set_xlim(-0.5, 1.5)
    axe.set_title(titre, fontsize=11)
    styliser(axe)
axes[0, 0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Test : 18 modèles (points : graines ; trait : moyenne ; barre : intervalle de confiance à 95 % de la moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "performances_test.png")
plt.show()
plt.close(fig)

# Figure 2 : estimations et intervalles des hypothèses, regroupées par unité.
groupes_figure = [
    ("H1 : F1 macro, CBAM − classique", ["H1"]),
    ("H2 et H3 : radios NORMAL (points de %)", ["H2", "H3"]),
    ("H4 : dépendance au format (ρ de Spearman)", ["H4 partie 1", "H4 partie 2"]),
]
fig, axes = plt.subplots(3, 1, figsize=(12, 9.5), gridspec_kw={"height_ratios": [3, 2, 2]})
for axe, (titre, codes_h) in zip(axes, groupes_figure):
    lignes_h = HYPOTHESES_RESULTATS[HYPOTHESES_RESULTATS["hypothese"].isin(codes_h)].reset_index(drop=True)
    for rang, ligne in lignes_h.iterrows():
        y = len(lignes_h) - 1 - rang
        valeurs = [float(v) for v in ligne["valeurs_par_cas"].split(" ; ")]
        axe.scatter(valeurs, np.full(len(valeurs), y) + np.linspace(-0.12, 0.12, len(valeurs)), s=22, color=ENCRE_DISCRETE, zorder=2)
        axe.hlines(y, ligne["ic95_bas"], ligne["ic95_haut"], color=ENCRE_SECONDAIRE, linewidth=2, zorder=3)
        axe.scatter([ligne["estimation"]], [y], s=70, color=ENCRE, zorder=4)
    axe.axvline(0, color=TRAIT_ZERO, linewidth=1, zorder=1)
    axe.set_yticks(range(len(lignes_h)), [f"{l['comparaison']}\nsens prédit : {l['sens_predit']} → {l['verdict']}"
                                          for _, l in lignes_h.iloc[::-1].iterrows()], fontsize=9)
    axe.set_ylim(-0.6, len(lignes_h) - 0.4)
    axe.set_title(titre, fontsize=11, loc="left")
    axe.grid(axis="x", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)
fig.suptitle("Hypothèses sur le test : estimation (point noir), intervalle à 95 % (trait) et valeur de chaque paire (points gris)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "hypotheses_test.png")
plt.show()
plt.close(fig)

## 4. Pourquoi ces résultats ? Comparaison du test avec les données de développement

Ces analyses sont **descriptives** : elles aident à interpréter les verdicts, elles n'en ajoutent pas.

1. **Décalage de distribution** : pour chaque classe, on compare le test aux données de développement (apprentissage + validation) sur le format d'origine, l'intensité et la netteté. L'AUC « test contre développement » mesure le décalage : 0,5 = même distribution, 1 = distributions séparées.
2. **Les raccourcis marchent-ils encore sur le test ?** Trois petits modèles logistiques, du même type qu'aux notebooks 11 et 12, sont entraînés sur l'apprentissage et évalués sur la validation puis sur le test : format seul et intensité v2 seule (comme au notebook 11), intensité et netteté de l'entrée v3 réunies (le notebook 12 les testait séparément). Si le format prédit bien la classe en validation mais mal sur le test, un modèle qui s'appuie sur le format se trompera davantage sur le test.
3. **Erreurs selon le format** : taux de faux positifs parmi les radios NORMAL et de faux négatifs parmi les radios PNEUMONIA, par tiers du rapport largeur/hauteur (tiers calculés sur le test, pour chaque classe).

In [ ]:
# 1. Caractéristiques du développement (notebooks 11 et 12) et du test (calculées plus haut avec les mêmes fonctions).
caracteristiques_v2 = pd.read_csv(DOSSIER_11 / "caracteristiques_images.csv", dtype={"groupe": str})
entrees_v3 = pd.read_csv(DOSSIER_12 / "caracteristiques_entrees_v3.csv", dtype={"groupe": str})
entrees_v3 = entrees_v3[["ensemble", "chemin_relatif", "moyenne", "ecart_type", "p05", "p50", "p95", "nettete"]].rename(
    columns={c: f"v3_{c}" for c in ["moyenne", "ecart_type", "p05", "p50", "p95", "nettete"]})
DEVELOPPEMENT = caracteristiques_v2.merge(entrees_v3, on=["ensemble", "chemin_relatif"], validate="one_to_one")
exiger(len(DEVELOPPEMENT) == len(train_df) + len(val_df), "Caractéristiques de développement incomplètes.")
TOUTES = pd.concat([DEVELOPPEMENT, CARACTERISTIQUES_TEST[DEVELOPPEMENT.columns]], ignore_index=True)

VARIABLES_DECALAGE = {"rapport_largeur_hauteur": "rapport largeur/hauteur d'origine", "hauteur": "hauteur d'origine (px)",
                      "largeur": "largeur d'origine (px)", "moyenne": "intensité moyenne (entrée v2)",
                      "ecart_type": "contraste (entrée v2)", "v3_ecart_type": "contraste (entrée v3)", "v3_nettete": "netteté (entrée v3)"}
lignes = []
for classe in ["NORMAL", "PNEUMONIA"]:
    developpement = TOUTES[(TOUTES["ensemble"] != "test") & (TOUTES["classe"] == classe)]
    test = TOUTES[(TOUTES["ensemble"] == "test") & (TOUTES["classe"] == classe)]
    for variable, libelle in VARIABLES_DECALAGE.items():
        a, b = developpement[variable].to_numpy(dtype=float), test[variable].to_numpy(dtype=float)
        auc = roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(b))], np.r_[a, b])
        lignes.append({"classe": classe, "variable": libelle, "mediane_developpement": np.median(a),
                       "q1_developpement": np.percentile(a, 25), "q3_developpement": np.percentile(a, 75),
                       "mediane_test": np.median(b), "q1_test": np.percentile(b, 25), "q3_test": np.percentile(b, 75),
                       "auc_test_contre_developpement": max(auc, 1 - auc),
                       "valeurs_plus_elevees_dans": "test" if auc > 0.5 else ("développement" if auc < 0.5 else "aucun")})
DECALAGE = pd.DataFrame(lignes)

# 2. Petits modèles « raccourcis » entraînés sur l'apprentissage, évalués sur la validation et le test.
GROUPES_VARIABLES = {
    "Format d'origine seul": ["largeur", "hauteur", "rapport_largeur_hauteur", "fraction_padding"],
    "Intensité de l'entrée v2 seule": ["moyenne", "ecart_type", "p05", "p50", "p95", "moyenne_centre", "moyenne_bords"],
    "Intensité et netteté de l'entrée v3": ["v3_moyenne", "v3_ecart_type", "v3_p05", "v3_p50", "v3_p95", "v3_nettete"],
}
lignes = []
apprentissage = TOUTES[TOUTES["ensemble"] == "train"]
for nom_groupe, variables in GROUPES_VARIABLES.items():
    modele_simple = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000))
    modele_simple.fit(apprentissage[variables].to_numpy(dtype=float), (apprentissage["classe"] == "PNEUMONIA").to_numpy(dtype=int))
    for ensemble in ["val", "test"]:
        tableau = TOUTES[TOUTES["ensemble"] == ensemble]
        y = (tableau["classe"] == "PNEUMONIA").to_numpy(dtype=int)
        probabilites = modele_simple.predict_proba(tableau[variables].to_numpy(dtype=float))[:, 1]
        prediction = (probabilites >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
        lignes.append({"informations_utilisees": nom_groupe, "ensemble": "validation" if ensemble == "val" else "test",
                       "auc": roc_auc_score(y, probabilites), "f1_macro": f1_score(y, prediction, average="macro"),
                       "specificite_NORMAL": tn / (tn + fp), "rappel_pneumonie": tp / (tp + fn)})
RACCOURCIS = pd.DataFrame(lignes)

# 3. Erreurs selon le format, sur le test.
LIBELLES_TIERS = ["format le plus étroit", "format intermédiaire", "format le plus large"]
tiers = {}
for classe in [0, 1]:
    bornes = np.quantile(RAPPORTS_TEST[Y_TEST == classe], [1 / 3, 2 / 3])
    tiers[classe] = np.digitize(RAPPORTS_TEST, bornes)
lignes = []
for cle in MODELES_ORDRE:
    prediction = PREDICTIONS[cle]["prediction"].to_numpy()
    for classe, erreur in [(0, "faux positifs (radios NORMAL)"), (1, "faux négatifs (radios PNEUMONIA)")]:
        for rang, libelle in enumerate(LIBELLES_TIERS):
            masque = (Y_TEST == classe) & (tiers[classe] == rang)
            lignes.append({"version": cle[0], "modele": cle[1], "graine": cle[2], "erreur": erreur, "tiers_format": libelle,
                           "n_images": int(masque.sum()), "taux_erreur_pct": 100 * float((prediction[masque] != classe).mean())})
ERREURS_FORMAT = pd.DataFrame(lignes)
RESUME_ERREURS_FORMAT = (ERREURS_FORMAT.groupby(["erreur", "version", "modele", "tiers_format"], sort=False)["taux_erreur_pct"]
                         .mean().unstack("tiers_format")[LIBELLES_TIERS].reset_index())

ecrire_csv(DECALAGE, SORTIE / "decalage_developpement_test.csv")
ecrire_csv(RACCOURCIS, SORTIE / "raccourcis_validation_test.csv")
ecrire_csv(ERREURS_FORMAT, SORTIE / "erreurs_par_format_test.csv")
ecrire_csv(RESUME_ERREURS_FORMAT, SORTIE / "resume_erreurs_par_format_test.csv")
print("Décalage entre développement et test, par classe (AUC : 0,5 = même distribution) :")
display(DECALAGE[["classe", "variable", "mediane_developpement", "mediane_test", "auc_test_contre_developpement",
                  "valeurs_plus_elevees_dans"]].round(3))
print("Petits modèles « raccourcis » : entraînés sur l'apprentissage, évalués sur la validation puis le test :")
display(RACCOURCIS.round(3))
print("Taux d'erreur (%) par tiers de format sur le test, moyenne sur 3 graines :")
display(RESUME_ERREURS_FORMAT.round(1))

# Figure 3 : format d'origine, développement contre test (fonctions de répartition).
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for colonne, classe in enumerate(["NORMAL", "PNEUMONIA"]):
    for rang, (variable, libelle) in enumerate([("rapport_largeur_hauteur", "Rapport largeur/hauteur d'origine"),
                                                ("hauteur", "Hauteur d'origine (pixels)")]):
        axe = axes[rang, colonne]
        for ensembles, couleur, epaisseur, nom_serie in [(["train", "val"], COULEUR_DEVELOPPEMENT, 1.8, "développement (apprentissage + validation)"),
                                                         (["test"], COULEUR_TEST, 2.4, "test")]:
            valeurs = np.sort(TOUTES[TOUTES["ensemble"].isin(ensembles) & (TOUTES["classe"] == classe)][variable].to_numpy(dtype=float))
            axe.step(valeurs, np.arange(1, len(valeurs) + 1) / len(valeurs), where="post", color=couleur, linewidth=epaisseur,
                     label=nom_serie)
        axe.set_title(f"Radios {classe} : {libelle.lower()}", fontsize=11)
        axe.set_ylabel("Part des radios (cumulée)")
        axe.set_ylim(0, 1.02)
        styliser(axe)
axes[0, 0].legend(fontsize=9, frameon=False, loc="lower right")
fig.suptitle("Format des radios d'origine : développement (gris) et test (violet)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "format_developpement_test.png")
plt.show()
plt.close(fig)

# Figure 4 : faux positifs parmi les radios NORMAL selon le format, sur le test.
fp = ERREURS_FORMAT[ERREURS_FORMAT["erreur"] == "faux positifs (radios NORMAL)"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
for axe, nom in zip(axes, ARCHITECTURES_NOMS):
    for version in VERSIONS_MODELES:
        sous = fp[(fp["version"] == version) & (fp["modele"] == nom)]
        moyennes = sous.groupby("tiers_format", sort=False)["taux_erreur_pct"].mean().reindex(LIBELLES_TIERS)
        decalage = {V2_SANS: -0.08, V2_AVEC: 0.0, V3: 0.08}[version]
        for k, libelle in enumerate(LIBELLES_TIERS):
            valeurs = sous[sous["tiers_format"] == libelle]["taux_erreur_pct"].to_numpy(dtype=float)
            axe.scatter(np.full(len(valeurs), k + decalage), valeurs, s=22, color=COULEURS[version], alpha=0.55, zorder=2)
        axe.plot(np.arange(3) + decalage, moyennes.to_numpy(), color=COULEURS[version], linewidth=2, marker="o",
                 markersize=8, markeredgecolor="white", markeredgewidth=1.5, zorder=3, label=version)
    axe.set_xticks(range(3), LIBELLES_TIERS)
    axe.set_title(LIBELLES_MODELES[nom], fontsize=11)
    styliser(axe)
axes[0].set_ylabel("Radios NORMAL classées PNEUMONIA (%)")
axes[0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Test : faux positifs selon le format d'origine (tiers calculés sur les radios NORMAL du test ; points : graines ; ligne : moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "faux_positifs_par_format_test.png")
plt.show()
plt.close(fig)

ecrire_json({"id_protocole": ID_PROTOCOLE, "effectif": N_TEST, "patients": N_PATIENTS,
             "verdicts": {f"{l['hypothese']} | {l['comparaison']}": l["verdict"] for l in BILAN_HYPOTHESES.to_dict("records")},
             "verdict_H4_ensemble": VERDICT_H4}, SORTIE / "termine.json")
FICHIERS_A_ENVOYER = ["bilan_hypotheses.csv", "hypotheses_resultats.csv", "details_hypotheses.csv", "synthese_test.csv",
                      "intervalles_test.csv", "mesures_test_par_modele.csv", "resume_validation_et_test.csv",
                      "decalage_developpement_test.csv", "raccourcis_validation_test.csv", "resume_erreurs_par_format_test.csv",
                      "termine.json"]
archive = SORTIE / "a_envoyer_notebook_14.zip"
temporaire = archive.with_suffix(".tmp")
with zipfile.ZipFile(temporaire, "w", compression=zipfile.ZIP_DEFLATED) as fichier_zip:
    for nom_tableau in FICHIERS_A_ENVOYER:
        fichier_zip.write(SORTIE / nom_tableau, arcname=nom_tableau)
    for figure in sorted((SORTIE / "figures").glob("*.png")):
        fichier_zip.write(figure, arcname=f"figures/{figure.name}")
remplacer(temporaire, archive)
print("Terminé. Pour transmettre les résultats, un seul fichier suffit (tableaux et figures) :")
print(" ", archive)

## Lecture des résultats

**H1, la question du mémoire.** Lire les trois versions ensemble ; v3 est la comparaison la plus juste, car elle se fait sans le raccourci principal. « Pas d'écart démontré » est un résultat valable : il s'écrira « sur le test, le CBAM n'améliore pas de façon démontrée le F1 macro ». Les mesures secondaires (AUC, rappel, spécificité) décrivent la nature de l'écart : une AUC identique avec une spécificité différente signifie que les deux modèles séparent aussi bien les classes, mais ne placent pas leur décision au même endroit.

**H2 et H3** disent si la nouvelle préparation (v3) et l'augmentation réduisent les faux positifs sur le test, la faiblesse observée au notebook 03.

**H4** confirme le mécanisme du raccourci seulement si ses deux parties sont confirmées : les modèles v2 se trompent davantage sur les radios NORMAL larges, et v3 beaucoup moins.

**Décalage entre développement et test.** Si les radios du test diffèrent du développement par le format ou l'intensité, et si le petit modèle « format seul » perd sa spécificité sur le test, alors un réseau qui s'appuie sur le format se trompe davantage sur le test. C'est l'explication à discuter pour l'effondrement de spécificité du notebook 03.

**Validation et test.** Un écart important entre validation et test signale une mauvaise généralisation. La validation a servi à choisir les époques et les réglages ; seul le test mesure la généralisation.

**Rapporter tous les verdicts, y compris ceux qui ne vont pas dans le sens attendu.** Les intervalles ne sont pas corrigés pour les comparaisons multiples ; trois graines seulement mesurent le hasard de l'entraînement.

**Suite :** envoyer `a_envoyer_notebook_14.zip`. Ensuite viendra la rédaction : chapitres des résultats et de la discussion, puis proposition d'application clinique (objectif 6 du sujet).